# 🧭 Financa — Financial-Behaviour Model
### AI Personal Finance Coach — behaviour forecasting (TensorFlow / Keras)

Trains a **sequence model** that reads a rolling window of a user's recent monthly
behaviour and predicts, for **next month**:

1. **Total spend** — as a *growth factor* on the user's current spend (regression).
2. **Overspend risk** — `P(next-month spend > budget)` (classification).

A shared **LSTM encoder → two heads** (Huber regression + sigmoid classification).

**Why a growth-factor target (not absolute money):** the model is **currency-invariant**.
Every input feature is a ratio/share/oscillator and the regression target is
`log(spend₊₁ / spendₜ)`, so the *identical* model serves a USD user and a Toman user —
the server just multiplies the predicted factor by that user's real current spend.

| Stage | What happens |
|---|---|
| Data | `datasets/Finance_dataset/finance_behavior.csv` (900 users × 30 months) |
| Features | 22 currency-invariant per-month features (spend/income, budget-util, savings-rate, spend-growth, weekend, largest-txn ratio, log n-txn, crypto intensity, month sin/cos, 11 category shares) |
| Sequences | lookback **L = 6** months → predict month 7; split by **user** (no leakage) |
| Model | LSTM(64) → LSTM(32) → Dense(32) → {reg head, cls head} |
| Eval | reg MAE vs. "no-change" baseline; cls accuracy/AUC vs. majority baseline; example predictions |
| Ship | `data/finance/finance_model.keras` · `finance_model_weights.h5` · `finance_scaler.pkl` · `finance_meta.json` |

Set `SMOKE = True` (cell 2) for a fast reduced-epoch run that still produces valid artifacts.

In [1]:
# =============================================================================
# 1 · ENVIRONMENT
# =============================================================================
import os, json, pickle, random, warnings
from datetime import datetime, timezone
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow", tf.__version__)

TensorFlow 2.16.2


In [2]:
# =============================================================================
# 2 · CONFIG
# =============================================================================
SMOKE = False        # True → fast reduced-epoch run (still saves valid artifacts)

DATA_CSV   = "datasets/Finance_dataset/finance_behavior.csv"
ART_DIR    = "data/finance"
os.makedirs(ART_DIR, exist_ok=True)

LOOKBACK   = 6       # months of context
TRAIN_USER_FRAC = 0.8
MAX_EPOCHS = 6 if SMOKE else 120
PATIENCE   = 3 if SMOKE else 14
BATCH      = 128

SPEND_CATS = ["food_dining", "groceries", "transport", "housing_utilities", "health",
              "entertainment", "shopping", "education", "subscriptions", "transfers", "other"]

# Feature order — MUST match the server's finance feature builder (finance_meta.json).
FEATURE_NAMES = ["spend_income", "budget_util", "savings_rate", "spend_growth",
                 "weekend_ratio", "largest_ratio", "log_ntxn", "crypto_intensity",
                 "crypto_txn", "month_sin", "month_cos"] + [f"share_{c}" for c in SPEND_CATS]
print(f"{len(FEATURE_NAMES)} features · lookback {LOOKBACK} · SMOKE={SMOKE}")

22 features · lookback 6 · SMOKE=False


In [3]:
# =============================================================================
# 3 · FEATURE ENGINEERING  (currency-invariant; identical formula used at serving)
# =============================================================================
def month_feature_row(cur, prev):
    """One month → feature vector. `cur`/`prev` are dict-likes with the CSV columns.
    Everything here is a ratio / share / oscillator — no absolute currency amount."""
    s = float(cur["spend"]); inc = float(cur["income"]); ps = float(prev["spend"])
    row = [
        np.clip(s / (inc + 1e-6), 0, 3),                       # spend-to-income
        np.clip(float(cur["budget_util"]), 0, 3),              # spend / budget
        np.clip(float(cur["savings_rate"]), -2, 1),            # savings rate
        np.clip(np.log((s + 1e-6) / (ps + 1e-6)), -1.5, 1.5),  # month-over-month spend growth
        float(cur["weekend_ratio"]),
        np.clip(float(cur["largest_txn"]) / (s + 1e-6), 0, 1), # concentration
        np.log1p(float(cur["n_transactions"])),                # count (currency-free)
        np.tanh(float(cur["crypto_value"]) / (s + 1e-6) / 2),  # crypto intensity (bounded)
        np.clip(float(cur["crypto_txn"]) / 10.0, 0, 2),
        np.sin(2 * np.pi * float(cur["month_of_year"]) / 12),
        np.cos(2 * np.pi * float(cur["month_of_year"]) / 12),
    ]
    for c in SPEND_CATS:
        row.append(np.clip(float(cur[f"cat_{c}"]) / (s + 1e-6), 0, 1))   # category shares
    return np.asarray(row, dtype=np.float32)


def build_user_matrix(g):
    """User rows (sorted) → (months, F) feature matrix."""
    g = g.sort_values("month_index").reset_index(drop=True)
    recs = g.to_dict("records")
    return np.stack([month_feature_row(recs[i], recs[i - 1] if i > 0 else recs[i])
                     for i in range(len(recs))]), g


df = pd.read_csv(DATA_CSV)
assert {"user_id", "month_index", "spend", "budget"}.issubset(df.columns)
print(f"loaded {len(df):,} rows · {df.user_id.nunique()} users")

# split by USER (no user appears in both train and val)
users = np.array(sorted(df.user_id.unique()))
rng = np.random.default_rng(SEED); rng.shuffle(users)
n_train = int(len(users) * TRAIN_USER_FRAC)
train_users, val_users = set(users[:n_train].tolist()), set(users[n_train:].tolist())

def make_sequences(user_ids):
    X, y_reg, y_cls = [], [], []
    for uid, g in df[df.user_id.isin(user_ids)].groupby("user_id"):
        M, gs = build_user_matrix(g)
        spend = gs["spend"].to_numpy(); over = gs["overspend"].to_numpy()
        for i in range(LOOKBACK - 1, len(gs) - 1):            # anchor i → predict i+1
            X.append(M[i - LOOKBACK + 1: i + 1])
            y_reg.append(np.clip(np.log((spend[i + 1] + 1e-6) / (spend[i] + 1e-6)), -1.5, 1.5))
            y_cls.append(float(over[i + 1]))
    return (np.asarray(X, np.float32), np.asarray(y_reg, np.float32), np.asarray(y_cls, np.float32))

Xtr, ytr_reg, ytr_cls = make_sequences(train_users)
Xva, yva_reg, yva_cls = make_sequences(val_users)
print(f"train seqs {Xtr.shape} · val seqs {Xva.shape}")
print(f"overspend base rate — train {ytr_cls.mean():.3f} · val {yva_cls.mean():.3f}")

loaded 27,000 rows · 900 users
train seqs (17280, 6, 22) · val seqs (4320, 6, 22)
overspend base rate — train 0.627 · val 0.658


In [4]:
# =============================================================================
# 4 · SCALING  (fit on train only; reg target standardized, bounds saved for inverse)
# =============================================================================
feat_scaler = StandardScaler().fit(Xtr.reshape(-1, Xtr.shape[-1]))
def scale(X): return feat_scaler.transform(X.reshape(-1, X.shape[-1])).reshape(X.shape).astype(np.float32)
Xtr_s, Xva_s = scale(Xtr), scale(Xva)

reg_mean, reg_std = float(ytr_reg.mean()), float(ytr_reg.std() + 1e-8)
ytr_reg_s = ((ytr_reg - reg_mean) / reg_std).astype(np.float32)
yva_reg_s = ((yva_reg - reg_mean) / reg_std).astype(np.float32)
print(f"reg target (log-growth): mean {reg_mean:+.4f} std {reg_std:.4f}")

reg target (log-growth): mean -0.0003 std 0.3740


In [5]:
# =============================================================================
# 5 · MODEL — shared LSTM encoder, two heads
# =============================================================================
def build_model(lookback, n_features):
    inp = keras.Input(shape=(lookback, n_features), name="months")
    x = keras.layers.LSTM(64, return_sequences=True, name="lstm_1")(inp)
    x = keras.layers.Dropout(0.25)(x)
    x = keras.layers.LSTM(32, name="lstm_2")(x)
    x = keras.layers.Dropout(0.25)(x)
    x = keras.layers.Dense(32, activation="swish", name="shared")(x)
    reg = keras.layers.Dense(1, name="spend_growth")(x)               # log-growth (standardized)
    cls = keras.layers.Dense(1, activation="sigmoid", name="overspend")(x)
    model = keras.Model(inp, [reg, cls])
    try:
        opt = keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4, clipnorm=1.0)
    except (AttributeError, TypeError):
        opt = keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0)
    model.compile(optimizer=opt,
                  loss={"spend_growth": keras.losses.Huber(1.0),
                        "overspend": keras.losses.BinaryCrossentropy()},
                  loss_weights={"spend_growth": 1.0, "overspend": 1.0},
                  metrics={"spend_growth": ["mae"], "overspend": ["accuracy", keras.metrics.AUC(name="auc")]})
    return model

model = build_model(LOOKBACK, len(FEATURE_NAMES))
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ months (InputLayer) │ (None, 6, 22)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 6, 64)     │     22,272 │ months[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 6, 64)     │          0 │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ (None, 32)        │     12,416 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 32)        │          0 │ lstm_2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared (Dense)      │ (None, 32)        │      1,056 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spend_growth        │ (None, 1)         │         33 │ shared[0][0]      │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ overspend (Dense)   │ (None, 1)         │         33 │ shared[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 35,810 (139.88 KB)

 Trainable params: 35,810 (139.88 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
# =============================================================================
# 6 · TRAIN
# =============================================================================
MODEL_PATH = os.path.join(ART_DIR, "finance_model.keras")
cbs = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True, verbose=1),
       keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=max(2, PATIENCE//2), min_lr=1e-5, verbose=0),
       keras.callbacks.ModelCheckpoint(MODEL_PATH, monitor="val_loss", save_best_only=True, verbose=0)]

hist = model.fit(Xtr_s, {"spend_growth": ytr_reg_s, "overspend": ytr_cls},
                 validation_data=(Xva_s, {"spend_growth": yva_reg_s, "overspend": yva_cls}),
                 epochs=MAX_EPOCHS, batch_size=BATCH, callbacks=cbs, shuffle=True, verbose=2)
model.save(MODEL_PATH)
best = int(np.argmin(hist.history["val_loss"])) + 1
print(f"best epoch {best}/{len(hist.history['val_loss'])} · val_loss {min(hist.history['val_loss']):.4f}")

Epoch 1/120
135/135 - 7s - 49ms/step - loss: 0.9857 - overspend_accuracy: 0.6619 - overspend_auc: 0.6607 - overspend_loss: 0.6242 - spend_growth_loss: 0.3615 - spend_growth_mae: 0.6883 - val_loss: 0.8476 - val_overspend_accuracy: 0.7053 - val_overspend_auc: 0.7276 - val_overspend_loss: 0.5708 - val_spend_growth_loss: 0.2754 - val_spend_growth_mae: 0.5821 - learning_rate: 0.0010
Epoch 2/120
135/135 - 3s - 21ms/step - loss: 0.8633 - overspend_accuracy: 0.6872 - overspend_auc: 0.7137 - overspend_loss: 0.5926 - spend_growth_loss: 0.2708 - spend_growth_mae: 0.5782 - val_loss: 0.7920 - val_overspend_accuracy: 0.7118 - val_overspend_auc: 0.7516 - val_overspend_loss: 0.5561 - val_spend_growth_loss: 0.2342 - val_spend_growth_mae: 0.5262 - learning_rate: 0.0010
Epoch 3/120
135/135 - 3s - 21ms/step - loss: 0.8377 - overspend_accuracy: 0.6928 - overspend_auc: 0.7260 - overspend_loss: 0.5834 - spend_growth_loss: 0.2543 - spend_growth_mae: 0.5547 - val_loss: 0.7808 - val_overspend_accuracy: 0.7118 -

In [7]:
# =============================================================================
# 7 · EVALUATE  (vs. honest baselines)
# =============================================================================
pred_reg_s, pred_cls = model.predict(Xva_s, verbose=0)
pred_growth = pred_reg_s.ravel() * reg_std + reg_mean          # inverse-standardize
true_growth = yva_reg

mae_model = float(np.mean(np.abs(pred_growth - true_growth)))
mae_naive = float(np.mean(np.abs(true_growth)))               # baseline: predict "no change" (growth 0)
# reconstructed next-month spend error (relative), model vs naive
rel_model = float(np.mean(np.abs(np.expm1(pred_growth) - np.expm1(true_growth))))
rel_naive = float(np.mean(np.abs(np.expm1(np.zeros_like(true_growth)) - np.expm1(true_growth))))

cls_p = pred_cls.ravel()
acc = float(((cls_p > 0.5).astype(float) == yva_cls).mean())
majority = float(max(yva_cls.mean(), 1 - yva_cls.mean()))
try:
    from sklearn.metrics import roc_auc_score
    auc = float(roc_auc_score(yva_cls, cls_p))
except Exception:
    auc = float("nan")

print("=== REGRESSION (next-month spend growth) ===")
print(f"  log-growth MAE : model {mae_model:.4f}  vs  no-change baseline {mae_naive:.4f}")
print(f"  spend-ratio MAE: model {rel_model:.4f}  vs  no-change baseline {rel_naive:.4f}")
print("=== CLASSIFICATION (overspend next month) ===")
print(f"  accuracy {acc:.3f}  vs majority {majority:.3f}   |   AUC {auc:.3f}")

metrics = {"reg_loggrowth_mae": round(mae_model, 4), "reg_loggrowth_mae_baseline": round(mae_naive, 4),
           "reg_spendratio_mae": round(rel_model, 4), "reg_spendratio_mae_baseline": round(rel_naive, 4),
           "cls_accuracy": round(acc, 4), "cls_majority_baseline": round(majority, 4), "cls_auc": round(auc, 4),
           "val_overspend_rate": round(float(yva_cls.mean()), 4)}

=== REGRESSION (next-month spend growth) ===
  log-growth MAE : model 0.1887  vs  no-change baseline 0.2571
  spend-ratio MAE: model 0.2014  vs  no-change baseline 0.2728
=== CLASSIFICATION (overspend next month) ===
  accuracy 0.714  vs majority 0.658   |   AUC 0.760


In [8]:
# =============================================================================
# 8 · EXAMPLE PREDICTIONS
# =============================================================================
for j in np.random.default_rng(0).integers(0, len(Xva_s), 5):
    g = float(pred_growth[j]); p = float(cls_p[j])
    print(f"  user-month {int(j):5d} | predicted spend ×{np.exp(g):.3f} "
          f"(actual ×{np.exp(true_growth[j]):.3f}) | overspend P={p:.2f} (actual {int(yva_cls[j])})")

  user-month  3674 | predicted spend ×0.672 (actual ×1.032) | overspend P=0.71 (actual 1)
  user-month  2751 | predicted spend ×1.018 (actual ×1.233) | overspend P=0.36 (actual 1)
  user-month  2208 | predicted spend ×0.668 (actual ×0.481) | overspend P=0.53 (actual 0)
  user-month  1165 | predicted spend ×1.015 (actual ×1.014) | overspend P=0.88 (actual 1)
  user-month  1329 | predicted spend ×1.006 (actual ×0.939) | overspend P=0.23 (actual 0)


In [10]:
# =============================================================================
# 9 · SAVE ARTIFACTS  (model + weights + scaler + metadata for the server)
# =============================================================================
model.save_weights(os.path.join(ART_DIR, "finance_model.weights.h5"))
with open(os.path.join(ART_DIR, "finance_scaler.pkl"), "wb") as fh:
    pickle.dump({"feature_scaler": feat_scaler, "feature_names": FEATURE_NAMES,
                 "spend_cats": SPEND_CATS, "lookback": LOOKBACK,
                 "reg_mean": reg_mean, "reg_std": reg_std}, fh)

meta = {"model_name": "financa_behaviour_lstm",
        "created_at": datetime.now(timezone.utc).isoformat(),
        "tensorflow_version": tf.__version__, "smoke_run": SMOKE,
        "lookback": LOOKBACK, "n_features": len(FEATURE_NAMES), "feature_names": FEATURE_NAMES,
        "spend_cats": SPEND_CATS, "reg_mean": reg_mean, "reg_std": reg_std,
        "tasks": {"regression": "log(next_spend / current_spend)",
                  "classification": "P(next_spend > next_budget)"},
        "metrics": metrics, "best_epoch": best,
        "n_train_seq": int(len(Xtr_s)), "n_val_seq": int(len(Xva_s)),
        "caveats": ["Trained on a synthetic-but-coherent behaviour panel; treat as decision "
                    "support. Features are currency-invariant so the same model serves USD & Toman "
                    "users — the server multiplies the predicted growth factor by the user's real spend."]}
with open(os.path.join(ART_DIR, "finance_meta.json"), "w", encoding="utf-8") as fh:
    json.dump(meta, fh, indent=2, ensure_ascii=False)

# reload check
re = keras.models.load_model(MODEL_PATH)
a, b = re.predict(Xva_s[:4], verbose=0); c, d = model.predict(Xva_s[:4], verbose=0)
assert np.allclose(a, c, atol=1e-5) and np.allclose(b, d, atol=1e-5)
print("saved + reload-verified →", ART_DIR)
print(json.dumps(metrics, indent=2))

saved + reload-verified → data/finance
{
  "reg_loggrowth_mae": 0.1887,
  "reg_loggrowth_mae_baseline": 0.2571,
  "reg_spendratio_mae": 0.2014,
  "reg_spendratio_mae_baseline": 0.2728,
  "cls_accuracy": 0.7141,
  "cls_majority_baseline": 0.6579,
  "cls_auc": 0.7595,
  "val_overspend_rate": 0.6579
}


## 10 · How the server uses this

`server.ipynb` loads `data/finance/finance_model.keras` + `finance_scaler.pkl` once, rebuilds the
**same 22 features** from the user's last 6 months of transaction JSON (via `finance_month_features`,
which mirrors `month_feature_row` above), and serves `GET /api/behavior`:

- **predicted next-month spend** = current real spend × `exp(predicted log-growth)` — in the user's own currency,
- **overspend probability** → low / medium / high risk,
- **grounded tips** derived from the drivers.

Because the target is a growth *factor* and every feature is scale-free, no re-training is needed
per currency or per user scale. Re-run this notebook (or after collecting more data) to refresh
the model; the server hot-path only reads the saved artifacts.